# Text Span JEPA + SIGReg · масштабируемое обучение без EMA

**Эксперимент 2.0 на DGX Spark.** Вместо маленького I-JEPA+EMA: обучаем **один энкодер с нуля**, общий для контекста и скрытого текстового блока. Он видит две непересекающиеся части текста; предиктор по видимой части предсказывает **один embedding скрытого блока**. Оба прохода получают градиент; **EMA, stop-gradient и отдельного учителя нет**. SIGReg предотвращает схлопывание распределения предсказанных и целевых эмбеддингов.

> **Научная оговорка:** это экспериментальная **адаптация идей LeJEPA** для предсказания текстовых spans, а не точная реплика опубликованной LeJEPA (где основным слагаемым служит согласование двух видов одного объекта). Преимущество SIGReg над EMA для *текста* не доказано; мы его исследуем.

**Данные**: FineWeb-Edu sample-10BT (потоковая загрузка, до 10 миллиардов GPT-2 токенов). **Настройка по умолчанию:** Transformer 12×768, примерно 125M параметров энкодера; профиль `large`: 16×1024, примерно 250M. Контекст 512 токенов, скрытый блок ~25%, обучающий бюджет по умолчанию 100M токенов с возможностью поднять до 500M/1B. Реальное число параметров выводится из кода. На одной DGX Spark скорость сильно зависит от режима и размера батча, поэтому сначала нужно измерить фактический throughput.

**Внимание:** впервые тестируйте на 3–5 шагах; точные скорости и устойчивость на DGX Spark не проверялись. GPU-память общая с другими процессами. Ни один токен GitHub и никакие модельные веса в GitHub не загружаются автоматически.

## 0. Зависимости
В рабочем окружении Spark нужен совместимый CUDA PyTorch. **Не переустанавливайте torch**, если он уже работает. Установка остальных зависимостей, при необходимости:
```bash
pip install "datasets>=3,<5" "transformers>=4.45,<6" matplotlib tqdm
```


In [ ]:
import os, math, random, time, json, copy, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch",torch.__version__,"device",DEVICE)
if DEVICE.type=="cuda":
    print(torch.cuda.get_device_name(0))
    torch.set_float32_matmul_precision("high")

## 1. Конфигурация

`base`: 12 слоёв, ширина 768. `large`: 16 слоёв, ширина 1024. Увеличение модели и корпуса — отдельные изменения; для чистой абляции не меняйте их одновременно. `max_steps` вычисляется из числа токенов и глобального батча.

In [ ]:
@dataclass
class Config:
    seed:int=42
    preset:str="base"  # "large"
    seq_len:int=512
    span_frac:float=0.25
    proj_dim:int=256
    micro_batch:int=8
    grad_accum:int=8
    token_budget:int=100_000_000  # для большого обучения: 500_000_000+
    lr:float=2e-4
    weight_decay:float=0.05
    sigreg_weight:float=0.10
    sigreg_directions:int=64
    dropout:float=0.0
    max_grad_norm:float=1.0
    eval_every:int=100
    save_every:int=500
    val_batches:int=8
    data_buffer:int=2048
    max_doc_tokens:int=4096
    out_dir:str="runs/text_span_sigreg_v2"
    hf_repo:str="HuggingFaceFW/fineweb-edu"
    hf_config:str="sample-10BT"
    tokenizer:str="gpt2"
cfg=Config()
if cfg.preset=="base":
    depth,width,heads=12,768,12
elif cfg.preset=="large":
    depth,width,heads=16,1024,16
else:
    raise ValueError("preset must be base or large")
assert 0<cfg.span_frac<0.5 and cfg.seq_len>32
assert cfg.micro_batch>=2 and cfg.grad_accum>=1
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(cfg.seed)
tokens_per_step=cfg.micro_batch*cfg.grad_accum*cfg.seq_len
max_steps=math.ceil(cfg.token_budget/tokens_per_step)
print(f"{cfg.preset} depth={depth} d={width} heads={heads} steps={max_steps} tokens/step={tokens_per_step:,}")

## 2. FineWeb-Edu через streaming с непересекающимися train/val

Dataset `sample-10BT` скачивается постепенно; исходный корпус целиком в RAM не загружается. Split определяется стабильным SHA256 от `id` документа (или текста при отсутствии `id`): ~1% валидация, остальные — обучение. Два split не пересекаются по этому идентификатору. Это не полная семантическая дедупликация корпуса.

Обучаем на последовательностях в пределах одного документа. Короткие хвосты отбрасываются; поэтому `token_budget` — число токенов, **поданных энкодеру**, а не гарантированное число уникальных прочитанных текстовых токенов. При перезапуске с checkpoint поток исходных данных начинается заново; точное восстановление позиции stream пока не реализовано.

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(cfg.tokenizer)
def is_validation(ex):
    key=str(ex.get("id") or ex.get("text","")[:1000])
    val=int.from_bytes(hashlib.sha256(key.encode("utf-8")).digest()[:8],"big") % 100
    return val == 0
class FineWebBlocks(IterableDataset):
    def __init__(self,part,shuffle=False):
        assert part in ("train","val")
        self.part=part;self.shuffle=shuffle
    def __iter__(self):
        ds=load_dataset(cfg.hf_repo,cfg.hf_config,split="train",streaming=True)
        if self.shuffle:
            ds=ds.shuffle(seed=cfg.seed,buffer_size=cfg.data_buffer)
        for ex in ds:
            if is_validation(ex)!=(self.part=="val"):continue
            text=str(ex.get("text",""))
            if len(text)<30:continue
            ids=tokenizer(text,add_special_tokens=False,truncation=True,max_length=cfg.max_doc_tokens)["input_ids"]
            for i in range(0,len(ids)-cfg.seq_len+1,cfg.seq_len):
                yield torch.tensor(ids[i:i+cfg.seq_len],dtype=torch.long)
train_loader=DataLoader(FineWebBlocks("train",shuffle=True),batch_size=cfg.micro_batch,num_workers=0,pin_memory=DEVICE.type=="cuda")
val_loader=DataLoader(FineWebBlocks("val"),batch_size=cfg.micro_batch,num_workers=0,pin_memory=DEVICE.type=="cuda")
# Зафиксированные данные валидации; повторная оценка не включает новые документы.
val_batches=[]
for i,v in enumerate(val_loader):
    if i>=cfg.val_batches:break
    val_batches.append(v)
assert val_batches,"Validation stream returned no batches."
print("Cached val batches:",len(val_batches))
print("Example:",tokenizer.decode(val_batches[0][0][:64]))

## 3. Разделяем текст на контекст и скрытый блок

Для каждого текста выбираем непрерывный блок длиной `span_frac * seq_len`. Контекстный проход получает все остальные токены и **их исходные позиции**. Целевой проход использует **только текст скрытого блока**, но **тот же самый энкодер и те же параметры**.

Это важно: target не видит контекст, поэтому не может восстановить собственный embedding, просто проигнорировав скрытый блок. Но сами свойства латентного пространства ещё надо проверять.

In [ ]:
def split_span(x,randomize=True):
    b,l=x.shape
    span=max(2,round(l*cfg.span_frac))
    if randomize:
        start=torch.randint(0,l-span+1,(b,),device=x.device)
    else:
        start=(torch.arange(b,device=x.device)*37+11)%(l-span+1)
    pos=torch.arange(l,device=x.device)[None,:].expand(b,-1)
    mask=(pos>=start[:,None])&(pos<start[:,None]+span)
    vpos=pos[~mask].reshape(b,l-span)
    tpos=pos[mask].reshape(b,span)
    return x.gather(1,vpos),vpos,x.gather(1,tpos),tpos,start

## 4. Один общий Transformer, небольшой предиктор блока

Энкодер двунаправленный, без causal attention. Он не использует предварительно обученные веса GPT-2 — используется только его токенизатор. Предсказатель получает видимый контекст и положение скрытого блока; cross-attention извлекает из контекста информацию, необходимую для прогноза target embedding.

In [ ]:
class Backbone(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.word=nn.Embedding(vocab,width)
        self.position=nn.Embedding(cfg.seq_len,width)
        block=nn.TransformerEncoderLayer(width,heads,4*width,dropout=cfg.dropout,activation="gelu",batch_first=True,norm_first=True)
        self.layers=nn.TransformerEncoder(block,depth,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(width)
    def forward(self,ids,positions):
        x=self.word(ids)+self.position(positions)
        return self.norm(self.layers(x))

class SpanHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.project=nn.Sequential(nn.LayerNorm(width),nn.Linear(width,width),nn.GELU(),nn.Linear(width,cfg.proj_dim))
        self.query=nn.Embedding(cfg.seq_len,width)
        self.cross=nn.MultiheadAttention(width,heads,batch_first=True)
        self.pred_norm=nn.LayerNorm(width)
        self.pred_out=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,cfg.proj_dim))
    def forward(self,context_h,target_h,starts):
        z_target=self.project(target_h.mean(dim=1))
        q=self.query(starts)[:,None,:]
        attended,_=self.cross(q,context_h,context_h,need_weights=False)
        z_pred=self.pred_out(self.pred_norm((q+attended).squeeze(1)))
        return z_pred,z_target

encoder=Backbone(len(tokenizer)).to(DEVICE)
head=SpanHead().to(DEVICE)
params=list(encoder.parameters())+list(head.parameters())
print("encoder parameters:",f"{sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")
print("total trainable:",f"{sum(p.numel() for p in params)/1e6:.1f}M")
print("EMA target encoder: NONE; both branches share encoder parameters.")

## 5. SIGReg как основная защита от коллапса

Для каждого батча обучаются **оба** набора латентов: предсказанный `z_pred` и целевой `z_target`. Между ними минимизируется MSE, а SIGReg проверяет одномерные случайные проекции обоих наборов на близость к нормальному распределению.

[
L=L_{span}+\lambda (L_{SIGReg}(z_{pred})+L_{SIGReg}(z_{target}))/2.
]

SIGReg здесь использует приближение сравнения характеристических функций. Маленькие microbatch могут давать шумную статистику — увеличение `micro_batch` предпочтительнее, если хватает памяти. Это **не точная реализация канонической LeJEPA**, а исследовательская текстовая вариация.

In [ ]:
def sigreg(z,directions=64):
    z=z.float()
    axes=F.normalize(torch.randn(z.shape[-1],directions,device=z.device),dim=0)
    projections=z@axes
    ts=torch.linspace(-3,3,25,device=z.device)
    phase=projections[:,:,None]*ts[None,None,:]
    est_re=phase.cos().mean(dim=0)
    est_im=phase.sin().mean(dim=0)
    normal=torch.exp(-0.5*ts.square())[None,:]
    return ((est_re-normal).square()+est_im.square()).mean()

def training_forward(x, randomize=True):
    context_ids,cp,target_ids,tp,starts=split_span(x,randomize)
    context_h=encoder(context_ids,cp)
    target_h=encoder(target_ids,tp)
    pred,target=head(context_h,target_h,starts)
    loss_pred=F.mse_loss(pred.float(),target.float())
    loss_reg=0.5*(sigreg(pred,cfg.sigreg_directions)+sigreg(target,cfg.sigreg_directions))
    return loss_pred+cfg.sigreg_weight*loss_reg,loss_pred,loss_reg,pred,target

opt=torch.optim.AdamW(params,lr=cfg.lr,weight_decay=cfg.weight_decay)
@torch.no_grad()
def evaluate():
    encoder.eval();head.eval()
    rows=[]
    for cpu_x in val_batches:
        x=cpu_x.to(DEVICE)
        with torch.autocast("cuda",dtype=torch.bfloat16,enabled=DEVICE.type=="cuda"):
            loss,lp,lr,pred,target=training_forward(x,False)
        p,t=F.normalize(pred.float(),dim=-1),F.normalize(target.float(),dim=-1)
        sims=p@t.T
        acc=(sims.argmax(dim=-1)==torch.arange(len(sims),device=x.device)).float().mean()
        center=target.float()-target.float().mean(0)
        singular=torch.linalg.svdvals(center)
        prob=singular.square();prob=prob/prob.sum().clamp_min(1e-9)
        effrank=torch.exp(-(prob*(prob+1e-12).log()).sum())
        rows.append(dict(pred_loss=float(lp),sigreg=float(lr),at1=float(acc),
                         latent_std=float(target.float().std(0).mean()),rank=float(effrank)))
    return {key:float(np.mean([row[key] for row in rows])) for key in rows[0]}
print("Initial validation:",evaluate())

## 6. Обучение с BF16, градиентным накоплением, checkpoint и resume

Контролируем сколько **токенов реально обработано**, а не только шаги. Для строгого сравнения сохраняйте seed, профиль, контекст и бюджет токенов.

Обратите внимание: SIGReg считается на каждом microbatch, а не на полном эффективном батче после gradient accumulation. Это компромисс по памяти, который можно улучшить в следующей версии.

`CHECKPOINT_PATH` можно указать для возобновления модели/оптимизатора. Возобновление не восстанавливает точную позицию стрима; новые документы повторно начинают читаться с начала, что указано в отчёте.

In [ ]:
from contextlib import nullcontext
Path(cfg.out_dir).mkdir(parents=True,exist_ok=True)
checkpoint_path=Path(cfg.out_dir)/"latest.pt"
resume=False  # True, если хотите возобновить предыдущий запуск с теми же параметрами
start_step=0
history=[]
if resume:
    blob=torch.load(checkpoint_path,map_location="cpu",weights_only=False)
    assert blob["config"]["preset"]==cfg.preset and blob["config"]["seq_len"]==cfg.seq_len
    encoder.load_state_dict(blob["encoder"])
    head.load_state_dict(blob["head"])
    opt.load_state_dict(blob["optimizer"])
    start_step=blob["step"]
    history=blob.get("history",[])
    print("Resuming at step",start_step)

def save_checkpoint(step):
    tmp=checkpoint_path.with_suffix(".tmp")
    torch.save({"config":asdict(cfg),"step":step,
        "encoder":encoder.state_dict(),"head":head.state_dict(),
        "optimizer":opt.state_dict(),"history":history},tmp)
    tmp.replace(checkpoint_path)

iterator=iter(train_loader)
t0=time.time()
for step in tqdm(range(start_step+1,max_steps+1),desc="Text Span JEPA + SIGReg"):
    encoder.train();head.train()
    opt.zero_grad(set_to_none=True)
    sum_pred=0.;sum_reg=0.
    for micro in range(cfg.grad_accum):
        try: xb=next(iterator)
        except StopIteration:
            iterator=iter(train_loader);xb=next(iterator)
        x=xb.to(DEVICE,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.bfloat16,enabled=DEVICE.type=="cuda"):
            loss,loss_pred,loss_reg,_,_=training_forward(x,True)
        (loss/cfg.grad_accum).backward()
        sum_pred+=float(loss_pred.detach())/cfg.grad_accum
        sum_reg+=float(loss_reg.detach())/cfg.grad_accum
    gn=float(nn.utils.clip_grad_norm_(params,cfg.max_grad_norm))
    opt.step()
    if step==1 or step%cfg.eval_every==0 or step==max_steps:
        result={"step":step,"tokens_seen":step*tokens_per_step,
            "train_pred":sum_pred,"train_sigreg":sum_reg,"grad_norm":gn,"sec":time.time()-t0}
        result.update({"val_"+k:v for k,v in evaluate().items()})
        history.append(result)
        print(result)
    if step%cfg.save_every==0 or step==max_steps:
        save_checkpoint(step)
print("Done. Model saved locally:",checkpoint_path)

## 7. Графики, метрики и отчёт для публикации

Модельные веса остаются **только локально** в `runs/`. В GitHub публикуйте конфигурацию, скалярные метрики и графики. Данные не включают исходные тексты или веса модели. Не интерпретируйте снижение MSE отдельно от SIGReg и downstream тестов.

Обратите внимание, что текущий `val_at1` — retrieval внутри microbatch; для строгой оценки нужны отрицательные примеры с одинаковой скрытой позицией и независимая downstream классификация. Это следующих этапов задача.

In [ ]:
def export_metrics():
    out=Path(cfg.out_dir)
    out.mkdir(parents=True,exist_ok=True)
    (out/"config.json").write_text(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
    (out/"metrics.json").write_text(json.dumps({
        "method":"text span shared encoder + SIGReg, NO EMA",
        "summary":"self-supervised span-embedding prediction; not canonical LeJEPA",
        "tokens_seen":history[-1]["tokens_seen"] if history else 0,
        "history":history,
        "validation_hash_holdout":True,
        "note":"model trained with streaming data, stream cursor not persisted"
    },indent=2,ensure_ascii=False))
    fig,axs=plt.subplots(1,3,figsize=(13,4))
    for ax,k in zip(axs,["train_pred","val_pred_loss","val_at1"]):
        ax.plot([v["tokens_seen"]/1e6 for v in history],[v[k] for v in history],marker=".")
        ax.set_title(k);ax.set_xlabel("tokens seen (M)");ax.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(out/"training.png",dpi=140)
    plt.show()
    plt.close(fig)
    print("Exported:",out/"config.json",out/"metrics.json",out/"training.png")
    return out
export_dir=export_metrics()

## 8. Публикация результатов в GitHub (опционально)

Эта ячейка отправляет **только три отчётных файла** в отдельную папку ветки эксперимента, не добавляя notebook, веса или другие локальные файлы. Нужны настроенные `git user.name`, `git user.email`, GitHub SSH/HTTPS-авторизация. Может выполняться из Jupyter без сброса пользовательских изменений `.ipynb`. Работает в текущем репозитории `crystalcali/salm`.

In [ ]:
import subprocess
from datetime import datetime,timezone
EXPECTED_BRANCH="experiment/text-i-jepa-notebook-20261113"
def run_git(*args,root=None,timeout=90):
    p=subprocess.run(["git",*args],cwd=root,stdout=subprocess.PIPE,
                     stderr=subprocess.PIPE,text=True,timeout=timeout)
    if p.returncode:
        raise RuntimeError("git "+ " ".join(args)+" failed: "+p.stderr[-1000:])
    return p.stdout.strip()
def publish_sigreg_report():
    if not history: raise RuntimeError("Train first")
    root=Path(run_git("rev-parse","--show-toplevel")).resolve()
    if run_git("branch","--show-current",root=root)!=EXPECTED_BRANCH:
        raise RuntimeError("Open the experimental branch before publishing.")
    origin=run_git("remote","get-url","origin",root=root)
    if origin.rstrip("/").removesuffix(".git") not in (
        "https://github.com/crystalcali/salm",
        "git@github.com:crystalcali/salm",
        "ssh://git@github.com/crystalcali/salm"):
        raise RuntimeError(f"Unexpected origin: {origin}")
    if run_git("diff","--cached","--name-only",root=root):
        raise RuntimeError("Git index already has staged files; resolve before publishing")
    source=export_metrics()
    timestamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    rel=Path("results")/"text_span_sigreg_v2"/(timestamp+"_"+cfg.preset+"_"+str(cfg.seed))
    dest=root/rel;dest.mkdir(parents=True,exist_ok=False)
    for name in ("config.json","metrics.json","training.png"):
        (dest/name).write_bytes((source/name).read_bytes())
    paths=[str(rel/name) for name in ("config.json","metrics.json","training.png")]
    run_git("add","--",*paths,root=root)
    if set(run_git("diff","--cached","--name-only",root=root).splitlines())!=set(paths):
        run_git("reset","--",*paths,root=root)
        raise RuntimeError("Unexpected paths staged")
    run_git("commit","-m",f"results: text SIGReg {timestamp}",root=root)
    run_git("push","origin",f"HEAD:refs/heads/{EXPECTED_BRANCH}",root=root,timeout=120)
    url="https://github.com/crystalcali/salm/tree/"+EXPECTED_BRANCH+"/"+rel.as_posix()
    print("Published:",url)
    return url

# Выполнить только когда нужны опубликованные результаты:
# report_url=publish_sigreg_report()